# Extract and store MM data in the same manner as FEcrys

In [1]:
import openmm as mm
from openmm import app
from simtk import unit
import mdtraj as md
import numpy as np
import pickle
import os
import sys
sys.path.append("../..")

In [2]:
# Constants
state = ("FCC", 60, 50)
state_name = "FCC_60K_50KP"

CONST_kB = 1e-3*8.31446261815324 # kilojoule/(kelvin*mole)
beta = 1 / (CONST_kB * state[1])
nonbonded_cutoff = 0.8  # nm
inp_plat = "CUDA"

ingro = "LJ_SMALLER.gro"
intop = "LJ_SMALLER.top"
intraj = "prod_long_npt.h5"

In [3]:
# Set simulation, read trajectory
grofile = os.path.join(ingro) # just to load the gro, we will erase the coordinates here.
topfile = os.path.join(intop)

# Load gro and top files
gro = app.GromacsGroFile(grofile)
top = app.GromacsTopFile(topfile, periodicBoxVectors=gro.getPeriodicBoxVectors()) 
integrator = mm.LangevinIntegrator(state[1]*unit.kelvin, 1.0/unit.picoseconds, 5.0*unit.femtoseconds)  # shouldn't matter, we are not integrating.
system = top.createSystem(nonbondedMethod=app.LJPME,nonbondedCutoff=nonbonded_cutoff*unit.nanometers) # should be consistent with LJ params used to generate
platform = mm.Platform.getPlatformByName(inp_plat)
simulation = app.Simulation(top.topology, system, integrator, platform)

# load existing trajectory
trajfile = os.path.join(intraj)
trajectory = md.load_hdf5(trajfile)

### MD Dataset data:

stride_save_frame : input

Can be read from trajectory:
* xyz
* b (box vectors)
* u (using simulation context)
* rb, but not v

Additional data that needs to be saved/gotten:
* velocities (v) - save during sim
* Temperatures (T) - save during sim
* COMs - calculated after sim

In [4]:
# Things that can be gotten easily
#     xyz
#     b (boxes)
#     u
r = simulation.context.getState(getPositions=True).getPositions(asNumpy=True)._value
xyz = trajectory.xyz
print(xyz.shape)
b = trajectory.unitcell_vectors
print(b.shape)
u = np.zeros(trajectory.n_frames)
for n in range(trajectory.n_frames):
    simulation.context.setPositions(trajectory.xyz[n])
    simulation.context.setPeriodicBoxVectors(a=b[n][0],b=b[n][1],c=b[n][2])
    u[n] = (simulation.context.getState(getEnergy=True).getPotentialEnergy()).value_in_unit(unit.kilojoule/unit.mole)
print(u.shape)
print(u[12000:12010])

(25000, 180, 3)
(25000, 3, 3)
(25000,)
[606.8985334  578.37224957 560.29502657 641.92448457 602.84289006
 590.05140637 642.15977938 655.58752889 631.36627408 626.67420488]


In [ ]:
# How to calculate COMs?
# All particles have same mass since only LJ spheres (see below)
masses = np.array([system.getParticleMass(i)._value for i in range(system.getNumParticles())])
print(masses)
mass_weights = masses / masses.sum()  # Called "_mu_" in FEcrys
print(mass_weights)

[16.043 16.043 16.043 16.043 16.043 16.043 16.043 16.043 16.043 16.043
 16.043 16.043 16.043 16.043 16.043 16.043 16.043 16.043 16.043 16.043
 16.043 16.043 16.043 16.043 16.043 16.043 16.043 16.043 16.043 16.043
 16.043 16.043 16.043 16.043 16.043 16.043 16.043 16.043 16.043 16.043
 16.043 16.043 16.043 16.043 16.043 16.043 16.043 16.043 16.043 16.043
 16.043 16.043 16.043 16.043 16.043 16.043 16.043 16.043 16.043 16.043
 16.043 16.043 16.043 16.043 16.043 16.043 16.043 16.043 16.043 16.043
 16.043 16.043 16.043 16.043 16.043 16.043 16.043 16.043 16.043 16.043
 16.043 16.043 16.043 16.043 16.043 16.043 16.043 16.043 16.043 16.043
 16.043 16.043 16.043 16.043 16.043 16.043 16.043 16.043 16.043 16.043
 16.043 16.043 16.043 16.043 16.043 16.043 16.043 16.043 16.043 16.043
 16.043 16.043 16.043 16.043 16.043 16.043 16.043 16.043 16.043 16.043
 16.043 16.043 16.043 16.043 16.043 16.043 16.043 16.043 16.043 16.043
 16.043 16.043 16.043 16.043 16.043 16.043 16.043 16.043 16.043 16.043
 16.04

In [30]:
# Get center of mass of frame
r_12000 = trajectory.xyz[12000]
print(r_12000.shape)

mu = mass_weights[:,np.newaxis]
print(mu.shape)
reduced_12000 = r_12000 * mu
print(reduced_12000.shape)
print(r_12000[5])
print(reduced_12000[5])

# Get COM
com_12000 = reduced_12000.sum(axis=0, keepdims=True)
print(com_12000)

(180, 3)
(180, 1)
(180, 3)
[ 0.33050436  0.5373579  -0.02990243]
[ 0.00183614  0.00298532 -0.00016612]
[[0.7813471  0.91021966 0.7064392 ]]


### Things that are inputs

args_initialise_object
* **PDB** : inputs/read
* **n_atoms_mol** : input
* **name** : input
* **FF_name** : name
* **atom_order_PDB_match_itp** : default false
* **FF_class** : default LJ class object (see sc_system.LJ)

args_initialise_system
* **PME_cutoff** : input
* **removeCMMotion** : default True
* **nonbondedMethod** : default LJPME (for LJ crystals)
* **custom_EwaldErrorTolerance** : input
* **constraints** : default None
* **SwitchingFunction_factor** : default 0.95 in FEcrys (want to ignore, set to 1)

args_initialise_simulation
* **rbv** : default None
* **minimise** : default True
* **T** : input
* **timestep_ps** : default (could be an input)
* **collision_rate** : default
* **P** : input
* **barostat_type** : default (numerical value for barostat, where 0=MC 1=Aniso MC 2= Flexible MC)
* **barostat_1_scaling** : default True (for anisotropic/flexible barostats)
* **stride_barostat** : default 25
* **custom_integrator** : default None

## Look at known data structure

In [19]:
mm_data_path = "../../O/MM/molecules/A/data/A_LJ_0.8_0.8_NVT_dataset_Form_fcc_Cell_smaller_Temp_60"

with open(mm_data_path, 'rb') as f:
    mm_data = pickle.load(f)

In [27]:
print(type(mm_data))
print(mm_data.keys())
print(mm_data['MD dataset'].keys())

<class 'dict'>
dict_keys(['MD dataset', 'args_initialise_object', 'args_initialise_system', 'args_initialise_simulation'])
dict_keys(['xyz', 'COMs', 'b', 'u', 'T', 'rbv', 'stride_save_frame'])


In [68]:
print(mm_data['args_initialise_object']['FF_name'])

LJ


In [26]:
for key in mm_data.keys():
    print(key)
    for innerkey in mm_data[key].keys():
        print(f"     {innerkey} : {type(mm_data[key][innerkey])}")
    print('')

MD dataset
     xyz : <class 'numpy.ndarray'>
     COMs : <class 'numpy.ndarray'>
     b : <class 'numpy.ndarray'>
     u : <class 'numpy.ndarray'>
     T : <class 'numpy.ndarray'>
     rbv : <class 'list'>
     stride_save_frame : <class 'int'>

args_initialise_object
     PDB : <class 'str'>
     n_atoms_mol : <class 'int'>
     name : <class 'str'>
     FF_name : <class 'str'>
     atom_order_PDB_match_itp : <class 'bool'>
     FF_class : <class 'type'>

args_initialise_system
     PME_cutoff : <class 'float'>
     removeCMMotion : <class 'bool'>
     nonbondedMethod : <class 'openmm.app.forcefield.LJPME'>
     custom_EwaldErrorTolerance : <class 'float'>
     constraints : <class 'NoneType'>
     SwitchingFunction_factor : <class 'float'>

args_initialise_simulation
     rbv : <class 'NoneType'>
     minimise : <class 'bool'>
     T : <class 'int'>
     timestep_ps : <class 'float'>
     collision_rate : <class 'int'>
     P : <class 'NoneType'>
     barostat_type : <class 'int'>
 

In [51]:
print(f"xyz: {mm_data['MD dataset']['xyz'].shape}")  # xyz coordinates by frame
print(f"COMs: {mm_data['MD dataset']['COMs'].shape}")  # System center of mass by frame
print(f"boxes: {mm_data['MD dataset']['b'].shape}")  # Box vectors in full matrix format I think
print(f"T: {mm_data['MD dataset']['T'].shape}")
print(f"u: {mm_data['MD dataset']['u'].shape}")
print(f"rbv: {len(mm_data['MD dataset']['rbv'])}")
print(f"{mm_data['MD dataset']['rbv'][0].shape}")
print(f"{mm_data['MD dataset']['rbv'][1].shape}")
print(f"{mm_data['MD dataset']['rbv'][2].shape}")
print(f"stide_save_frame: {mm_data['MD dataset']['stride_save_frame']}")

xyz: (200000, 180, 3)
COMs: (200000, 1, 3)
boxes: (200000, 3, 3)
T: (200000,)
u: (200000, 1)
rbv: 3
(180, 3)
(3, 3)
(180, 3)
stide_save_frame: 50


In [ ]:
print(mm_data['args_initialise_object']['FF_class'])

<class 'O.MM.sc_system.LJ'>


In [63]:
# Check rbv
"""
r = last set of positions
b = last set of box vectors/matrix
v = last set of velocities
"""

#print(f"{mm_data['MD dataset']['rbv'][0]}")
#print(f"{mm_data['MD dataset']['rbv'][1]}")
#print(f"{mm_data['MD dataset']['rbv'][2]}")

'\nr = last set of positions\nb = last set of box vectors/matrix\nv = last set of velocities\n'

In [64]:
print(mm_data['args_initialise_object']['PDB'])

./O//MM/molecules/A/A_LJ_0.8_0.8_equilibrated_Form_fcc_Cell_smaller_Temp_60.pdb
